In [10]:
import csv

# 1. Leer productos y asignar id incremental (el orden es el id)
products_file = '../data/products.csv'
products = {}
with open(products_file, newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)  # saltar encabezado
    for idx, row in enumerate(reader, start=1):
        code = row[0].strip()
        products[code] = idx  # id incremental

# 2. Leer relación contrato-producto
relacion = {}
with open('../data/contrato_producto.csv', newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)  # saltar encabezado
    for row in reader:
        contrato = row[0].strip()
        clave = row[1].strip()
        if clave:
            relacion[contrato] = clave

# 3. Procesar créditos y agregar product_id y credit_status
with open('../data/credits.csv', newline='', encoding='utf-8') as fin, \
     open('../data/credits_con_product_id.csv', 'w', newline='', encoding='utf-8') as fout:
    reader = csv.DictReader(fin)
    fieldnames = list(reader.fieldnames)
    if 'product_id' not in fieldnames:
        fieldnames.append('product_id')
    if 'credit_status' not in fieldnames:
        fieldnames.append('credit_status')
    writer = csv.DictWriter(fout, fieldnames=fieldnames)
    writer.writeheader()
    for row in reader:
        contrato = row['contract_number'].strip()
        clave = relacion.get(contrato)
        if clave and clave in products:
            row['product_id'] = products[clave]
        else:
            row['product_id'] = 8  # DEFAULT
        row['credit_status'] = 'defaulted'
        writer.writerow(row)

print("Archivo credits_con_product_id.csv generado con éxito.")

Archivo credits_con_product_id.csv generado con éxito.


In [12]:
import csv
import psycopg2
from datetime import datetime

def parse_decimal(value):
    if not value or value.strip() == '':
        return None
    return float(value.strip())

def parse_date(value):
    if not value or value.strip() == '':
        return None
    return datetime.strptime(value.strip(), '%Y-%m-%d').date()

def parse_null(value):
    if not value or value.strip() == '' or value.strip() == '0':
        return None
    return value.strip()

def map_payment_frequency(value):
    if not value:
        return None
    v = value.strip().upper()
    if v == 'SEMANAL':
        return 'weekly'
    if v == 'QUINCENAL':
        return 'biweekly'
    if v == 'MENSUAL':
        return 'monthly'
    if v == 'TRIMESTRAL':
        return 'quarterly'
    return None  # o puedes retornar v.lower() si quieres dejar pasar otros valores

# Configura tus datos de conexión
DB_HOST = '69.48.206.219'
DB_PORT = '5432'
DB_NAME = 'collection_db'
DB_USER = 'cobranza'
DB_PASS = 'cobranza2025'

def main():
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )
    cur = conn.cursor()

    with open('../data/credits_con_product_id.csv', newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            cur.execute("""
                INSERT INTO collection.credits (
                    investment_id, debtor_id, contract_number, opening_date,
                    utilized_amount, total_financed, number_of_payments, payment_frequency,
                    payment_amount, missed_payment_date, current_balance, outstanding_capital,
                    ordinary_interest, collection_expense_amount, balance_date,
                    product_id, market_type, contract_type, credit_type,
                    beneficiary_name, beneficiary_relationship, credit_status
                ) VALUES (
                    %(investment_id)s, %(debtor_id)s, %(contract_number)s, %(opening_date)s,
                    %(utilized_amount)s, %(total_financed)s, %(number_of_payments)s, %(payment_frequency)s,
                    %(payment_amount)s, %(missed_payment_date)s, %(current_balance)s, %(outstanding_capital)s,
                    %(ordinary_interest)s, %(collection_expense_amount)s, %(balance_date)s,
                    %(product_id)s, %(market_type)s, %(contract_type)s, %(credit_type)s,
                    %(beneficiary_name)s, %(beneficiary_relationship)s, %(credit_status)s
                )
            """, {
                'investment_id': int(row['investment_id']),
                'debtor_id': int(row['debtor_id']),
                'contract_number': row['contract_number'],
                'opening_date': parse_date(row['opening_date']),
                'utilized_amount': parse_decimal(row['utilized_amount']),
                'total_financed': parse_decimal(row['total_financed']),
                'number_of_payments': int(row['number_of_payments']),
                'payment_frequency': map_payment_frequency(row['payment_frequency']),
                'payment_amount': parse_decimal(row['payment_amount']),
                'missed_payment_date': parse_date(row['missed_payment_date']),
                'current_balance': parse_decimal(row['current_balance']),
                'outstanding_capital': parse_decimal(row['outstanding_capital']),
                'ordinary_interest': parse_decimal(row['ordinary_interest']),
                'collection_expense_amount': parse_decimal(row['collection_expense_amount']),
                'balance_date': parse_date(row['balance_date']),
                'product_id': int(row['product_id']) if row['product_id'] else None,
                'market_type': row['market_type'] if row['market_type'] else None,
                'contract_type': row['contract_type'] if row['contract_type'] else None,
                'credit_type': row['credit_type'] if row['credit_type'] else None,
                'beneficiary_name': parse_null(row['beneficiary_name']),
                'beneficiary_relationship': parse_null(row['beneficiary_relationship']),
                'credit_status': row['credit_status'] if row['credit_status'] else None
            })
    conn.commit()
    cur.close()
    conn.close()
    print("Datos insertados correctamente.")

if __name__ == '__main__':
    main()

Datos insertados correctamente.
